In [ ]:
%pip install --ignore-installed -r path/to/requirements_container.txt

In [ ]:
from __future__ import annotations

import ast
import contextlib
import csv
import json
import math
import os
import queue
import re
import socket
import subprocess
import sys
import textwrap
import threading
import time
import urllib.error
import urllib.request
from dataclasses import asdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from typing import Sequence

import pandas as pd
from jupyter_client import KernelManager
from openai import OpenAI
from transformers import AutoTokenizer

In [ ]:
HF_TOKEN = "your_hf_token"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

In [ ]:
class CFG:
    
    project_root = Path(
        os.environ.get(
            "AIMO_PROJECT_ROOT",
            str(Path.cwd()),
        )
    ).resolve()

    requirements_path = Path(
        os.environ.get(
            "AIMO_REQUIREMENTS_PATH",
            "path/to/requirements_container.txt",
        )
    ).expanduser().resolve()

    eval_parquet_path = Path(
        os.environ.get(
            "AIMO_EVAL_PARQUET_PATH",
            "path/to/aimo_proof_eval.parquet",
        )
    ).expanduser().resolve()

    venv_path = Path(
        os.environ.get(
            "AIMO_VENV_PATH",
            str(project_root / "venv-container"),
        )
    ).resolve()

    venv_bin_path = venv_path / "bin"

    configured_venv_python_path = Path(
        os.environ.get(
            "AIMO_VENV_PYTHON_PATH",
            str(venv_bin_path / "python"),
        )
    ).resolve()

    if configured_venv_python_path.exists() and configured_venv_python_path.parent == venv_bin_path:
        venv_python_path = configured_venv_python_path
    elif (venv_bin_path / "python").exists():
        venv_python_path = (venv_bin_path / "python").resolve()
    else:
        venv_python_path = Path(sys.executable).resolve()

    model_path = os.environ.get(
        "AIMO_MODEL_PATH",
        "allenai/Olmo-3.1-32B-Think",
    )

    served_model_name = os.environ.get(
        "AIMO_SERVED_MODEL_NAME",
        "OLMo-3.1-32B-Think",
    )

    trust_remote_code = (
        os.environ.get("AIMO_TRUST_REMOTE_CODE", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    host = os.environ.get(
        "AIMO_HOST",
        "127.0.0.1",
    )

    server_port = int(
        os.environ.get(
            "AIMO_PORT",
            "8000",
        )
    )

    base_url = f"http://{host}:{server_port}/v1"
    api_key = "sk-local"

    launch_server = (
        os.environ.get("AIMO_LAUNCH_SERVER", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    reuse_server = (
        os.environ.get("AIMO_REUSE_SERVER", "false").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    output_root = Path(
        os.environ.get(
            "AIMO_OUTPUT_ROOT",
            str(project_root / "output" / "runpod"),
        )
    ).resolve()

    output_path = Path(
        os.environ.get(
            "AIMO_OUTPUT_TXT",
            str(output_root / "aimo_proof_outputs.txt"),
        )
    ).resolve()

    log_path = Path(
        os.environ.get(
            "AIMO_LOG_PATH",
            str(output_root / "aimo_notebook_status.json"),
        )
    ).resolve()

    cache_root = Path(
        os.environ.get(
            "AIMO_CACHE_ROOT",
            str(output_root / "cache"),
        )
    ).resolve()

    hf_home = Path(
        os.environ.get(
            "HF_HOME",
            str(cache_root / "huggingface"),
        )
    )

    transformers_cache = Path(
        os.environ.get(
            "TRANSFORMERS_CACHE",
            str(hf_home / "transformers"),
        )
    )

    huggingface_hub_cache = Path(
        os.environ.get(
            "HUGGINGFACE_HUB_CACHE",
            str(hf_home / "hub"),
        )
    )

    pip_cache = Path(
        os.environ.get(
            "PIP_CACHE_DIR",
            str(cache_root / "pip"),
        )
    )

    torch_home = Path(
        os.environ.get(
            "TORCH_HOME",
            str(cache_root / "torch"),
        )
    )

    triton_cache = Path(
        os.environ.get(
            "TRITON_CACHE_DIR",
            str(cache_root / "triton"),
        )
    )

    vllm_cache = Path(
        os.environ.get(
            "VLLM_CACHE_ROOT",
            str(cache_root / "vllm"),
        )
    )

    jupyter_config_dir = Path(
        os.environ.get(
            "JUPYTER_CONFIG_DIR",
            str(output_root / "jupyter" / "config"),
        )
    )

    jupyter_data_dir = Path(
        os.environ.get(
            "JUPYTER_DATA_DIR",
            str(output_root / "jupyter" / "data"),
        )
    )

    jupyter_runtime_dir = Path(
        os.environ.get(
            "JUPYTER_RUNTIME_DIR",
            str(output_root / "jupyter" / "runtime"),
        )
    )

    seed = int(
        os.environ.get(
            "AIMO_SEED",
            "42",
        )
    )

    quick_run = (
        os.environ.get("AIMO_QUICK_RUN", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    quick_run_problem_count = int(
        os.environ.get(
            "AIMO_QUICK_RUN_PROBLEM_COUNT",
            "2",
        )
    )

    quick_run_seed = int(
        os.environ.get(
            "AIMO_QUICK_RUN_SEED",
            str(seed),
        )
    )

    problem_limit = int(
        os.environ.get(
            "AIMO_PROBLEM_LIMIT",
            "0",
        )
    )

    max_model_len = int(
        os.environ.get(
            "AIMO_MAX_MODEL_LEN",
            "65536",
        )
    )

    max_generation_tokens = int(
        os.environ.get(
            "AIMO_MAX_GENERATION_TOKENS",
            "57344",
        )
    )

    minimum_available_tokens = int(
        os.environ.get(
            "AIMO_MINIMUM_AVAILABLE_TOKENS",
            "512",
        )
    )

    tensor_parallel_size = int(
        os.environ.get(
            "AIMO_TENSOR_PARALLEL_SIZE",
            "1",
        )
    )

    max_num_seqs = int(
        os.environ.get(
            "AIMO_MAX_NUM_SEQS",
            "8",
        )
    )

    max_num_batched_tokens = int(
        os.environ.get(
            "AIMO_MAX_NUM_BATCHED_TOKENS",
            "4096",
        )
    )

    max_logprobs = int(
        os.environ.get(
            "AIMO_MAX_LOGPROBS",
            "0",
        )
    )

    request_logprobs = int(
        os.environ.get(
            "AIMO_REQUEST_LOGPROBS",
            "0",
        )
    )

    stream_interval = int(
        os.environ.get(
            "AIMO_STREAM_INTERVAL",
            "128",
        )
    )

    enable_async_scheduling = (
        os.environ.get("AIMO_ENABLE_ASYNC_SCHEDULING", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    enable_prefix_caching = (
        os.environ.get("AIMO_ENABLE_PREFIX_CACHING", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    enable_chunked_prefill = (
        os.environ.get("AIMO_ENABLE_CHUNKED_PREFILL", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    disable_log_stats = (
        os.environ.get("AIMO_DISABLE_LOG_STATS", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    gpu_memory_utilization = float(
        os.environ.get(
            "AIMO_GPU_MEMORY_UTILIZATION",
            "0.95",
        )
    )

    compilation_config = {
        "cudagraph_capture_sizes": [
            1,
            2,
            4,
            8,
        ],
        "cudagraph_specialize_lora": False,
        "max_cudagraph_capture_size": 8,
    }

    temperature = float(
        os.environ.get(
            "AIMO_TEMPERATURE",
            "0.6",
        )
    )

    top_p = float(
        os.environ.get(
            "AIMO_TOP_P",
            "0.95",
        )
    )

    top_k = int(
        os.environ.get(
            "AIMO_TOP_K",
            "-1",
        )
    )

    min_p = float(
        os.environ.get(
            "AIMO_MIN_P",
            "0.0",
        )
    )

    presence_penalty = float(
        os.environ.get(
            "AIMO_PRESENCE_PENALTY",
            "0.0",
        )
    )

    repetition_penalty = float(
        os.environ.get(
            "AIMO_REPETITION_PENALTY",
            "1.0",
        )
    )

    stop_strings = [
        "<|im_end|>",
    ]

    request_timeout = float(
        os.environ.get(
            "AIMO_REQUEST_TIMEOUT_SECONDS",
            "1500",
        )
    )

    output_ping_interval = float(
        os.environ.get(
            "AIMO_OUTPUT_PING_SECONDS",
            "15",
        )
    )

    output_pulse_enabled = (
        os.environ.get("AIMO_ENABLE_OUTPUT_PULSE", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    cuda_preflight_enabled = (
        os.environ.get("AIMO_ENABLE_CUDA_PREFLIGHT", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    server_timeout = float(
        os.environ.get(
            "AIMO_SERVER_START_TIMEOUT_SECONDS",
            "900",
        )
    )

    jupyter_timeout = float(
        os.environ.get(
            "AIMO_JUPYTER_TIMEOUT_SECONDS",
            "10",
        )
    )

    pass_timeout = float(
        os.environ.get(
            "AIMO_PASS_TIMEOUT_SECONDS",
            "1500",
        )
    )

    tool_timeout = float(
        os.environ.get(
            "AIMO_TOOL_TIMEOUT_SECONDS",
            "10",
        )
    )

    sandbox_output_chars = int(
        os.environ.get(
            "AIMO_SANDBOX_OUTPUT_CHARS",
            "4096",
        )
    )

    max_tool_rounds = int(
        os.environ.get(
            "AIMO_MAX_TOOL_ROUNDS",
            "128",
        )
    )

    max_python_calls = int(
        os.environ.get(
            "AIMO_MAX_PYTHON_CALLS",
            "128",
        )
    )

    enable_vllm_auto_tool_choice = (
        os.environ.get("AIMO_ENABLE_VLLM_AUTO_TOOL_CHOICE", "true").strip().lower()
        in {"1", "true", "yes", "on"}
    )

    vllm_tool_call_parser = os.environ.get(
        "AIMO_VLLM_TOOL_CALL_PARSER",
        "olmo3",
    ).strip()

    vllm_chat_template_content_format = os.environ.get(
        "AIMO_VLLM_CHAT_TEMPLATE_CONTENT_FORMAT",
        "string",
    ).strip()

    problem_columns = (
        "problem",
        "problem_markdown",
        "statement",
        "prompt",
        "question",
    )

    reference_columns = (
        "solution",
        "solution_markdown",
        "reference_solution",
    )

    first_pass_system_prompt = textwrap.dedent(
        """
        You are one of the most capable contestants selected for this year's International Mathematical Olympiad, seated for the Contest and preparing a solution to be read by experienced olympiad graders. The work should have the form of a contest submission: a coherent argument in standard mathematical prose, with definitions introduced before they are used, notation kept stable, lemmas justified, cases separated cleanly, and the conclusion following without gaps or contradictions.

        Before the final submission is written, the available Python tool should be used for any calculation, finite check, symbolic experiment, or counterexample search that could strengthen the argument or expose a hidden error. The final answer must prove every step of the solution: every reduction, identity, inequality, case split, finiteness claim, monotonicity claim, divisibility claim, extremal claim, and appeal to a known theorem must be justified in the text before it is used. Computational evidence may guide the proof, but it cannot replace a proof. When the final solution is ready, close any <think> section and then write only the solution itself: no code, no tool output, no drafting notes, and no discussion of private reasoning.
        """
    ).strip()

    second_pass_system_prompt = textwrap.dedent(
        """
        You are serving on the International Mathematical Olympiad Jury for this year's contest, reviewing a proposed solution before an official solution is released. The review should follow the standards of coordination: locate every mathematical error, missing case, unjustified implication, inconsistent notation, or unsupported claim, and then produce a corrected solution suitable for publication.

        The final official solution should be written as polished mathematical prose in English. It should be text-only, self-contained, complete, step by step, and mathematically consistent. Before that final version is written, the available Python tool should be used wherever computations, examples, finite cases, symbolic identities, boundary cases, or possible counterexamples may affect correctness. If the proposed solution is flawed, replace the flawed argument; if it is sound, recast it as a clean official proof. The final answer must prove every step of the solution: every reduction, identity, inequality, case split, finiteness claim, monotonicity claim, divisibility claim, extremal claim, and appeal to a known theorem must be justified in the text before it is used. Computational evidence may guide the proof, but it cannot replace a proof. When the final official solution is ready, close any <think> section and then write only the solution itself: no code, no tool output, no grading notes, no false starts, and no discussion of private reasoning.
        """
    ).strip()

    tool_prompt = textwrap.dedent(
        """
        The python tool runs code in a stateful Jupyter kernel with a 10 second timeout per call. The environment provides math, statistics, random, collections, itertools, functools, fractions, decimal, sympy as sp, numpy as np, and mpmath as mp.

        Exact arithmetic with fractions or sympy is preferred for identities, integer problems, divisibility, modular checks, recurrences, and polynomial manipulation. Numerical packages such as numpy or mpmath are appropriate for exploration, but numerical evidence should be converted into exact reasoning before it appears in the final proof. Finite searches should report the search bounds, invariants checked, and any counterexamples or confirming summaries. Output should remain concise so the proof context stays readable.

        Calls are written one per line as python(code="...") inside <function_calls></function_calls>; multiline code is represented with escaped \\n characters inside the string.
        """
    ).strip()

    first_pass_prompt = textwrap.dedent(
        """
        Problem:
        {problem}

        A complete solution is required. Use the available Python tool before writing the final solution whenever computation can clarify arithmetic, examples, finite cases, symbolic identities, or possible counterexamples. The submitted answer should read like a contest solution: text-only, self-contained, step by step, and consistent in notation and cases. Prove every nontrivial assertion before relying on it; do not use phrases such as clearly, by computation, it is known, it follows, or this pattern continues unless the reason is supplied in the proof. Close any <think> section before writing the final submitted solution.
        """
    ).strip()

    second_pass_prompt = textwrap.dedent(
        """
        Problem:
        {problem}

        Proposed solution:
        {first_solution}

        A final official solution is required after critical review of the proposed solution. Use the available Python tool wherever computations, examples, finite cases, symbolic identities, boundary cases, or possible counterexamples may affect correctness. The published answer should repair every error, missing case, unsupported claim, or inconsistency, and should be a text-only, complete, step-by-step proof in polished mathematical prose. Prove every nontrivial assertion before relying on it; do not use phrases such as clearly, by computation, it is known, it follows, or this pattern continues unless the reason is supplied in the proof. Close any <think> section before writing the final official solution.
        """
    ).strip()

    vllm_environment = {
        "VLLM_DO_NOT_TRACK": "1",
        "VLLM_NO_USAGE_STATS": "1",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_FLAX": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
        "TORCH_COMPILE_DISABLE": "1",
        "TORCHDYNAMO_DISABLE": "1",
        "TORCHINDUCTOR_DISABLE": "1",
        "AIMO_DISABLE_TORCH_COMPILE": "1",
        "VLLM_LOG_STATS_INTERVAL": "60",
        "VLLM_USE_FLASHINFER_SAMPLER": "0",
        "VLLM_ENFORCE_STRICT_TOOL_CALLING": "1",
    }

    kernel_environment = {
        "PYDEVD_DISABLE_FILE_VALIDATION": "1",
        "PYDEVD_WARN_EVALUATION_TIMEOUT": "0",
        "JUPYTER_PLATFORM_DIRS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
        "PYTHONWARNINGS": "ignore",
        "MKL_NUM_THREADS": "1",
        "OMP_NUM_THREADS": "1",
        "MPLBACKEND": "Agg",
    }

In [ ]:
@dataclass(frozen=True)
class AIMOProblemRecord:
    
    order_index: int
    id: str
    problem: str
    reference_solution: str
    metadata: dict[str, str]


@dataclass(frozen=True)
class AIMOChatMessage:
    
    role: str
    content: str


@dataclass(frozen=True)
class AIMOToolCall:
    
    name: str
    code: str
    tool_call_id: str = ""


@dataclass(frozen=True)
class AIMOChatResponse:
    
    content: str
    tool_calls: list[AIMOToolCall]
    raw_tool_calls: list[dict[str, Any]]


@dataclass(frozen=True)
class AIMOToolResult:
    
    success: bool
    output: str
    error: str
    timed_out: bool

    def to_payload(self) -> str:
        
        if self.timed_out:
            return "Python execution timed out."

        if self.success:
            return self.output.strip() or "Python execution completed with no output."

        return self.error.strip() or "Python execution failed."


@dataclass(frozen=True)
class AIMOToolExecution:
    
    code: str
    output: str
    success: bool
    timed_out: bool


@dataclass(frozen=True)
class AIMOPassResult:
    
    name: str
    final_text: str
    tool_output: str
    python_calls: int
    python_errors: int
    elapsed_seconds: float
    input_tokens: int
    output_tokens: int


@dataclass(frozen=True)
class AIMOProblemResult:
    
    record: AIMOProblemRecord
    passes: list[AIMOPassResult]


class AIMOOutputPulse:
    
    def __init__(self, interval_seconds: float, enabled: bool) -> None:
        
        self.interval_seconds = interval_seconds
        self.enabled = enabled
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.dots_on_line = 0

    def __enter__(self) -> AIMOOutputPulse:
        
        if self.enabled and self.interval_seconds > 0:
            self.thread.start()

        return self

    def __exit__(self, exception_type: object, exception: object, traceback: object) -> None:
        
        self.stop()

    def run(self) -> None:
        
        while not self.stop_event.wait(self.interval_seconds):
            print(".", end="", flush=True)
            self.dots_on_line += 1

            if self.dots_on_line >= 10:
                print("", flush=True)
                self.dots_on_line = 0

    def stop(self) -> None:
        
        self.stop_event.set()

        if self.thread.is_alive():
            self.thread.join(timeout=1.0)

        if self.enabled:
            print("", flush=True)

In [ ]:
class AIMOEnvironment:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def configure(self) -> None:
        
        for path in self.cache_paths():
            path.mkdir(parents=True, exist_ok=True)

        self.cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.cfg.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.export_paths()
        self.validate_paths()

    def export_paths(self) -> None:
        
        os.environ["AIMO_PROJECT_ROOT"] = str(self.cfg.project_root)
        os.environ["AIMO_OUTPUT_ROOT"] = str(self.cfg.output_root)
        os.environ["AIMO_REQUIREMENTS_PATH"] = str(self.cfg.requirements_path)
        os.environ["AIMO_EVAL_PARQUET_PATH"] = str(self.cfg.eval_parquet_path)
        os.environ["AIMO_VENV_PATH"] = str(self.cfg.venv_path)
        os.environ["AIMO_VENV_PYTHON_PATH"] = str(self.cfg.venv_python_path)
        os.environ["HF_HOME"] = str(self.cfg.hf_home)
        os.environ["TRANSFORMERS_CACHE"] = str(self.cfg.transformers_cache)
        os.environ["HUGGINGFACE_HUB_CACHE"] = str(self.cfg.huggingface_hub_cache)
        os.environ["PIP_CACHE_DIR"] = str(self.cfg.pip_cache)
        os.environ["TORCH_HOME"] = str(self.cfg.torch_home)
        os.environ["TRITON_CACHE_DIR"] = str(self.cfg.triton_cache)
        os.environ["VLLM_CACHE_ROOT"] = str(self.cfg.vllm_cache)
        os.environ["JUPYTER_CONFIG_DIR"] = str(self.cfg.jupyter_config_dir)
        os.environ["JUPYTER_DATA_DIR"] = str(self.cfg.jupyter_data_dir)
        os.environ["JUPYTER_RUNTIME_DIR"] = str(self.cfg.jupyter_runtime_dir)
        os.environ.update(self.cfg.vllm_environment)

        if self.cfg.venv_python_path.parent == self.cfg.venv_bin_path:
            current_path = os.environ.get("PATH", "")
            os.environ["VIRTUAL_ENV"] = str(self.cfg.venv_path)
            os.environ["PATH"] = os.pathsep.join([
                str(self.cfg.venv_bin_path),
                current_path,
            ])

    def validate_paths(self) -> None:
        
        if not self.cfg.requirements_path.exists():
            raise FileNotFoundError(f"requirements file does not exist: {self.cfg.requirements_path}")

        if not self.cfg.eval_parquet_path.exists():
            raise FileNotFoundError(f"eval parquet does not exist: {self.cfg.eval_parquet_path}")

        if not self.cfg.eval_parquet_path.is_file():
            raise FileNotFoundError(f"eval parquet is not a file: {self.cfg.eval_parquet_path}")

        if not self.cfg.venv_python_path.exists():
            raise FileNotFoundError(f"venv python does not exist: {self.cfg.venv_python_path}")

    def cache_paths(self) -> list[Path]:
        
        return [
            self.cfg.cache_root,
            self.cfg.hf_home,
            self.cfg.transformers_cache,
            self.cfg.huggingface_hub_cache,
            self.cfg.pip_cache,
            self.cfg.torch_home,
            self.cfg.triton_cache,
            self.cfg.vllm_cache,
            self.cfg.jupyter_config_dir,
            self.cfg.jupyter_data_dir,
            self.cfg.jupyter_runtime_dir,
        ]


class AIMOInputReader:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def read_records(self) -> list[AIMOProblemRecord]:
        
        dataset_path = self.find_dataset_path()
        dataframe = pd.read_parquet(dataset_path)

        if self.cfg.quick_run:
            dataframe = self.quick_run_dataframe(dataframe)
        elif self.cfg.problem_limit > 0:
            dataframe = dataframe.head(self.cfg.problem_limit)

        records = [
            self.record_from_row(row_payload, order_index)
            for order_index, row_payload in enumerate(dataframe.to_dict(orient="records"))
        ]

        return records

    def quick_run_dataframe(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        
        sample_size = min(len(dataframe), max(0, self.cfg.quick_run_problem_count))

        if sample_size <= 0:
            return dataframe.head(0)

        sampled_dataframe = dataframe.sample(
            n=sample_size,
            random_state=self.cfg.quick_run_seed,
        ).sort_index()

        return sampled_dataframe

    def find_dataset_path(self) -> Path:
        
        path = self.cfg.eval_parquet_path

        if not path.exists():
            raise FileNotFoundError(f"eval parquet does not exist: {path}")

        if not path.is_file():
            raise FileNotFoundError(f"eval parquet is not a file: {path}")

        return path

    def record_from_row(self, row_payload: dict[str, object], order_index: int) -> AIMOProblemRecord:
        
        row = {
            str(key): value
            for key, value in row_payload.items()
        }
        problem = self.select_first(row, self.cfg.problem_columns)
        reference_solution = self.select_first(row, self.cfg.reference_columns)
        record_id = self.clean_value(row.get("id") or row.get("problem_id") or order_index)
        ignored_columns = {"id", "problem_id", *self.cfg.problem_columns, *self.cfg.reference_columns}
        metadata = {
            key: self.clean_value(value)
            for key, value in row.items()
            if key not in ignored_columns
        }

        if not problem.strip():
            raise ValueError(f"Problem text is empty for row {order_index}.")

        return AIMOProblemRecord(
            order_index=order_index,
            id=record_id,
            problem=problem,
            reference_solution=reference_solution,
            metadata=metadata,
        )

    def select_first(self, row: dict[str, object], columns: Sequence[str]) -> str:
        
        for column in columns:
            if column in row:
                value = self.clean_value(row[column]).strip()

                if value:
                    return value

        return ""

    def clean_value(self, value: object) -> str:
        
        if value is None:
            return ""

        try:
            if pd.isna(value):
                return ""
        except (TypeError, ValueError):
            pass

        return str(value)


class AIMOOutputWriter:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def write_results(self, results: list[AIMOProblemResult]) -> None:
        
        blocks = [
            self.format_problem_result(result)
            for result in results
        ]
        self.cfg.output_path.write_text(
            "\n\n".join(blocks).strip() + "\n",
            encoding="utf-8",
        )
        self.write_status(results)

    def write_status(self, results: list[AIMOProblemResult]) -> None:
        
        payload = {
            "problem_count": len(results),
            "output_path": str(self.cfg.output_path),
            "totals": self.metrics_totals(results),
            "problems": [
                {
                    "id": result.record.id,
                    "passes": [
                        {
                            "name": pass_result.name,
                            "python_calls": pass_result.python_calls,
                            "python_errors": pass_result.python_errors,
                            "elapsed_seconds": pass_result.elapsed_seconds,
                            "input_tokens": pass_result.input_tokens,
                            "generated_tokens": pass_result.output_tokens,
                            "total_tokens": pass_result.input_tokens + pass_result.output_tokens,
                        }
                        for pass_result in result.passes
                    ],
                }
                for result in results
            ],
        }
        self.cfg.log_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

    def metrics_dataframe(self, results: list[AIMOProblemResult]) -> pd.DataFrame:
        
        rows = []

        for result in results:
            for pass_index, pass_result in enumerate(result.passes, start=1):
                rows.append({
                    "problem_id": result.record.id,
                    "pass_index": pass_index,
                    "pass_name": pass_result.name,
                    "input_tokens": pass_result.input_tokens,
                    "generated_tokens": pass_result.output_tokens,
                    "total_tokens": pass_result.input_tokens + pass_result.output_tokens,
                    "python_calls": pass_result.python_calls,
                    "python_errors": pass_result.python_errors,
                    "elapsed_seconds": round(pass_result.elapsed_seconds, 2),
                })

        totals = self.metrics_totals(results)

        if rows:
            rows.append({
                "problem_id": "TOTAL",
                "pass_index": 0,
                "pass_name": "all",
                "input_tokens": totals["input_tokens"],
                "generated_tokens": totals["generated_tokens"],
                "total_tokens": totals["total_tokens"],
                "python_calls": totals["python_calls"],
                "python_errors": totals["python_errors"],
                "elapsed_seconds": round(totals["elapsed_seconds"], 2),
            })

        return pd.DataFrame(rows)

    def metrics_totals(self, results: list[AIMOProblemResult]) -> dict[str, int | float]:
        
        passes = [
            pass_result
            for result in results
            for pass_result in result.passes
        ]

        input_tokens = sum(pass_result.input_tokens for pass_result in passes)
        generated_tokens = sum(pass_result.output_tokens for pass_result in passes)

        return {
            "input_tokens": input_tokens,
            "generated_tokens": generated_tokens,
            "total_tokens": input_tokens + generated_tokens,
            "python_calls": sum(pass_result.python_calls for pass_result in passes),
            "python_errors": sum(pass_result.python_errors for pass_result in passes),
            "elapsed_seconds": sum(pass_result.elapsed_seconds for pass_result in passes),
        }

    def format_problem_result(self, result: AIMOProblemResult) -> str:
        
        sections = [
            "=" * 80,
            f"Problem ID: {result.record.id}",
            "",
            "Problem:",
            result.record.problem.strip(),
        ]

        for pass_index, pass_result in enumerate(result.passes, start=1):
            sections.extend([
                "",
                f"{pass_result.name.title()} Final Channel:",
                pass_result.final_text.strip() or "No final channel was produced.",
            ])

        return "\n".join(sections)

In [ ]:
class AIMOChatMLTemplate:
    
    function_call_pattern = re.compile(r"<function_calls>(.*?)</function_calls>", re.DOTALL)
    think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)
    code_block_pattern = re.compile(r"```(?:python|py)?\s*.*?```", re.IGNORECASE | re.DOTALL)
    executable_code_block_pattern = re.compile(r"```(?:python|py)\s*(.*?)```", re.IGNORECASE | re.DOTALL)
    im_tag_pattern = re.compile(r"<\|im_start\|>.*?|<\|im_end\|>", re.DOTALL)
    tool_call_opening = "<function_calls>\n"
    tool_call_closing = "</function_calls>"

    def __init__(self, tokenizer: Any) -> None:
        
        self.tokenizer = tokenizer

    def render_initial_prompt(self, messages: Sequence[AIMOChatMessage], tool_prompt: str) -> str:
        
        return self.render_messages(
            messages=messages,
            tool_prompt=tool_prompt,
            add_generation_prompt=True,
        )

    def render_messages(
        self,
        messages: Sequence[AIMOChatMessage],
        tool_prompt: str,
        add_generation_prompt: bool,
    ) -> str:
        
        rendered_messages = [
            self.message_payload(message)
            for message in messages
        ]
        tools = self.tool_schema(tool_prompt) if tool_prompt else None

        if tools is None:
            return str(
                self.tokenizer.apply_chat_template(
                    rendered_messages,
                    tokenize=False,
                    add_generation_prompt=add_generation_prompt,
                )
            )

        return str(
            self.tokenizer.apply_chat_template(
                rendered_messages,
                tools=tools,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
        )

    def wrap_tool_generation(self, text: str) -> str:
        
        stripped_text = text.strip()

        if "<function_calls>" in stripped_text:
            if "</function_calls>" in stripped_text:
                return stripped_text

            return f"{stripped_text}\n{self.tool_call_closing}"

        return f"{self.tool_call_opening}{stripped_text}\n{self.tool_call_closing}"

    def normalize_tool_text(self, text: str) -> str:
        
        if "<function_calls>" in text and "</function_calls>" not in text:
            return self.wrap_tool_generation(text)

        return text

    def message_payload(self, message: AIMOChatMessage) -> dict[str, Any]:
        
        return {
            "role": message.role,
            "content": self.escape_content(message.content),
        }

    def tool_schema(self, tool_prompt: str) -> list[dict[str, Any]]:
        
        return [
            {
                "type": "function",
                "function": {
                    "name": "python",
                    "description": tool_prompt,
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "code": {
                                "type": "string",
                                "description": "Python code to execute.",
                            },
                        },
                        "required": ["code"],
                        "additionalProperties": False,
                    },
                },
            },
        ]

    def parse_tool_calls(self, text: str) -> list[AIMOToolCall]:
        
        normalized_text = self.normalize_tool_text(text)
        calls = []

        for match in self.function_call_pattern.finditer(normalized_text):
            call_text = match.group(1).strip()
            json_calls = self.parse_json_calls(call_text)

            if json_calls:
                calls.extend(json_calls)
                continue

            for line in call_text.splitlines():
                stripped_line = line.strip()

                if not stripped_line:
                    continue

                call = self.parse_python_call(stripped_line)

                if call is not None:
                    calls.append(call)

        if calls:
            return calls

        calls.extend(self.parse_code_block_calls(normalized_text))

        if calls:
            return calls

        for line in normalized_text.splitlines():
            stripped_line = line.strip()

            if not stripped_line.startswith("python("):
                continue

            call = self.parse_python_call(stripped_line)

            if call is not None:
                calls.append(call)

        return calls

    def parse_code_block_calls(self, text: str) -> list[AIMOToolCall]:
        
        calls = []

        for match in self.executable_code_block_pattern.finditer(text):
            code = match.group(1).strip()

            if code:
                calls.append(AIMOToolCall(name="python", code=code))

        return calls

    def parse_json_calls(self, text: str) -> list[AIMOToolCall]:
        
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            return []

        items = payload if isinstance(payload, list) else [payload]
        calls = []

        for item in items:
            if not isinstance(item, dict):
                continue

            function_payload = item.get("function", {})

            if isinstance(function_payload, dict):
                name = str(item.get("name") or function_payload.get("name") or "")
                arguments = item.get("arguments", function_payload.get("arguments", {}))
            else:
                name = str(item.get("name") or function_payload or "")
                arguments = item.get("arguments", {})

            if isinstance(arguments, str):
                try:
                    arguments = json.loads(arguments)
                except json.JSONDecodeError:
                    arguments = {"code": arguments}

            if name != "python" or not isinstance(arguments, dict):
                continue

            code = str(arguments.get("code", "")).strip()

            if code:
                calls.append(AIMOToolCall(name="python", code=code))

        return calls

    def parse_python_call(self, text: str) -> AIMOToolCall | None:
        
        try:
            expression = ast.parse(text, mode="eval").body
        except SyntaxError:
            return None

        if not isinstance(expression, ast.Call):
            return None

        if not isinstance(expression.func, ast.Name) or expression.func.id != "python":
            return None

        code = self.code_from_call(expression)

        if not code:
            return None

        return AIMOToolCall(name="python", code=code)

    def code_from_call(self, expression: ast.Call) -> str:
        
        for keyword in expression.keywords:
            if keyword.arg == "code":
                try:
                    return str(ast.literal_eval(keyword.value)).strip()
                except (ValueError, TypeError):
                    return ""

        if expression.args:
            try:
                return str(ast.literal_eval(expression.args[0])).strip()
            except (ValueError, TypeError):
                return ""

        return ""

    def final_channel(self, text: str) -> str:
        
        without_tools = self.function_call_pattern.sub("", text)
        if "</think>" in without_tools:
            final_text = without_tools.split("</think>")[-1]
        elif "<think>" in without_tools:
            return "No final answer was produced."
        else:
            final_text = without_tools

        final_text = self.think_pattern.sub("", final_text)
        final_text = self.code_block_pattern.sub("", final_text)
        final_text = self.im_tag_pattern.sub("", final_text)
        final_text = final_text.replace("<think>", "")
        final_text = final_text.replace("</think>", "")

        return "\n".join(
            line.rstrip()
            for line in final_text.splitlines()
        ).strip()

    def escape_content(self, content: str) -> str:
        
        return (
            str(content)
            .replace("<|im_start|>", "<|escaped_im_start|>")
            .replace("<|im_end|>", "<|escaped_im_end|>")
        )

In [ ]:
class AIMOSandbox:
    
    _port_lock = threading.Lock()
    _blacklisted_ports = set()
    _last_blacklist_clear = time.time()
    _next_port = 50000

    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.client = None
        self.kernel_manager = None
        self.start_kernel()
        self.prime_kernel()

    def start_kernel(self) -> None:
        
        last_exception = None

        for retry_index in range(3):
            ports = None

            try:
                ports = self.next_ports(5)
                self.kernel_manager = KernelManager()
                self.kernel_manager.shell_port = ports[0]
                self.kernel_manager.iopub_port = ports[1]
                self.kernel_manager.stdin_port = ports[2]
                self.kernel_manager.hb_port = ports[3]
                self.kernel_manager.control_port = ports[4]
                self.kernel_manager.start_kernel(
                    env=self.kernel_environment(),
                    extra_arguments=["--Application.log_level=CRITICAL"],
                )
                self.client = self.kernel_manager.blocking_client()
                self.client.start_channels()
                self.client.wait_for_ready(timeout=self.cfg.jupyter_timeout)

                return
            except Exception as exception:
                last_exception = exception

                if ports and "Address already in use" in str(exception):
                    self.blacklist_ports(ports)

                if self.kernel_manager is not None:
                    with contextlib.suppress(Exception):
                        self.kernel_manager.shutdown_kernel(now=True)

                if retry_index < 2:
                    time.sleep(0.5 * (2 ** retry_index))

        raise RuntimeError(f"Failed to start Jupyter kernel: {last_exception}")

    def execute(self, code: str) -> AIMOToolResult:
        
        message_id = self.client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False,
        )
        stdout_parts = []
        stderr_parts = []
        started_at = time.time()

        while True:
            if time.time() - started_at > self.cfg.tool_timeout:
                self.kernel_manager.interrupt_kernel()

                return AIMOToolResult(
                    success=False,
                    output="",
                    error="Python execution timed out.",
                    timed_out=True,
                )

            try:
                message = self.client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if message.get("parent_header", {}).get("msg_id") != message_id:
                continue

            message_type = message.get("msg_type")
            content = message.get("content", {})

            if message_type == "stream":
                if content.get("name") == "stdout":
                    stdout_parts.append(str(content.get("text", "")))
                else:
                    stderr_parts.append(str(content.get("text", "")))
            elif message_type == "error":
                stderr_parts.append("\n".join(str(item) for item in content.get("traceback", [])))
            elif message_type in {"execute_result", "display_data"}:
                text = content.get("data", {}).get("text/plain")

                if text:
                    stdout_parts.append(str(text))
            elif message_type == "status" and content.get("execution_state") == "idle":
                break

        output = self.compact_text("".join(stdout_parts))
        error = self.compact_error("".join(stderr_parts))

        return AIMOToolResult(
            success=not bool(error),
            output=output,
            error=error,
            timed_out=False,
        )

    def prime_kernel(self) -> None:
        
        self.execute(
            "import collections\n"
            "import decimal\n"
            "import fractions\n"
            "import functools\n"
            "import itertools\n"
            "import math\n"
            "import random\n"
            "import statistics\n"
            "import numpy as np\n"
            "import sympy as sp\n"
            "import mpmath as mp\n"
            "from fractions import Fraction\n"
            "from decimal import Decimal, getcontext\n"
            "getcontext().prec = 64\n"
            "mp.mp.dps = 64\n"
        )

    def reset(self) -> None:
        
        with contextlib.suppress(Exception):
            self.execute("%reset -f")
            self.prime_kernel()

    def close(self) -> None:
        
        if self.client is not None:
            with contextlib.suppress(Exception):
                self.client.stop_channels()

        if self.kernel_manager is not None:
            with contextlib.suppress(Exception):
                self.kernel_manager.shutdown_kernel(now=True)

    def kernel_environment(self) -> dict[str, str]:
        
        environment = os.environ.copy()
        environment.update(self.cfg.kernel_environment)

        return environment

    def compact_error(self, error_text: str) -> str:
        
        clean_text = self.strip_ansi(error_text)
        lines = []

        for line in clean_text.splitlines():
            stripped_line = line.strip()

            if not stripped_line:
                continue

            if "Traceback (most recent call last)" in stripped_line:
                continue

            if stripped_line.startswith("File "):
                continue

            if stripped_line.startswith("^"):
                continue

            lines.append(stripped_line)

        return self.compact_text("\n".join(lines))

    def compact_text(self, text: str) -> str:
        
        clean_text = self.strip_ansi(text).strip()

        if len(clean_text) <= self.cfg.sandbox_output_chars:
            return clean_text

        slice_chars = self.cfg.sandbox_output_chars // 2
        omitted_chars = len(clean_text) - self.cfg.sandbox_output_chars

        return (
            f"{clean_text[:slice_chars]}\n"
            f"... [Truncated {omitted_chars} characters] ...\n"
            f"{clean_text[-slice_chars:]}"
        )

    def strip_ansi(self, text: str) -> str:
        
        return re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", text)

    @classmethod
    def next_ports(cls, count: int) -> list[int]:
        
        with cls._port_lock:
            cls.clear_old_blacklist()

            for _ in range(10):
                start_port = cls._next_port
                ports = []

                for offset in range(count):
                    port = start_port + offset

                    if port > 65535:
                        start_port = 50000
                        port = start_port + offset
                        ports = []

                    if cls.port_available(port):
                        ports.append(port)
                    else:
                        cls._next_port = port + 1
                        ports = []
                        break

                if len(ports) == count:
                    cls._next_port = ports[-1] + 1

                    return ports

                cls._next_port += 10

                if cls._next_port > 65535:
                    cls._next_port = 50000

        raise RuntimeError("Unable to find available Jupyter ports.")

    @classmethod
    def clear_old_blacklist(cls) -> None:
        
        current_time = time.time()

        if current_time - cls._last_blacklist_clear > 300:
            cls._blacklisted_ports.clear()
            cls._last_blacklist_clear = current_time

    @classmethod
    def blacklist_ports(cls, ports: list[int]) -> None:
        
        with cls._port_lock:
            cls._blacklisted_ports.update(ports)

    @classmethod
    def port_available(cls, port: int) -> bool:
        
        if port in cls._blacklisted_ports:
            return False

        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as socket_object:
                socket_object.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                socket_object.bind(("127.0.0.1", port))

                return True
        except OSError:
            return False

In [ ]:
class AIMOServer:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.process = None
        self.stdout_file = None
        self.stderr_file = None
        self.stdout_path = self.cfg.output_root / "vllm_stdout.log"
        self.stderr_path = self.cfg.output_root / "vllm_stderr.log"
        self.command_path = self.cfg.output_root / "vllm_command.json"

    def __enter__(self) -> AIMOServer:
        
        self.start()

        return self

    def __exit__(self, exception_type: object, exception: object, traceback: object) -> None:
        
        self.stop()

    def start(self) -> None:
        
        if self.cfg.reuse_server:
            self.wait_until_ready()
            return

        if not self.cfg.launch_server:
            return

        if not self.port_available(self.cfg.host, self.cfg.server_port):
            raise RuntimeError(f"Port is already in use: {self.cfg.host}:{self.cfg.server_port}")

        self.validate_cuda_runtime()

        command = self.build_command()
        self.command_path.write_text(
            json.dumps({"command": command, "health_url": self.health_url()}, indent=2) + "\n",
            encoding="utf-8",
        )
        self.stdout_file = self.stdout_path.open("w", encoding="utf-8")
        self.stderr_file = self.stderr_path.open("w", encoding="utf-8")
        self.process = subprocess.Popen(
            command,
            stdout=self.stdout_file,
            stderr=self.stderr_file,
            text=True,
            env=self.server_environment(),
        )
        self.wait_until_ready()

    def stop(self) -> None:
        
        if self.process is not None and self.process.poll() is None:
            self.process.terminate()

            try:
                self.process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=30)

        for file_object in [self.stdout_file, self.stderr_file]:
            if file_object is not None:
                file_object.close()

    def python_executable(self) -> Path:
        
        if not self.cfg.venv_python_path.exists():
            raise FileNotFoundError(f"vLLM server python does not exist: {self.cfg.venv_python_path}")

        return self.cfg.venv_python_path

    def validate_cuda_runtime(self) -> None:
        
        if not self.cfg.cuda_preflight_enabled:
            return

        script = (
            "import torch\n"
            "print(f'torch={torch.__version__}')\n"
            "print(f'torch_cuda={torch.version.cuda}')\n"
            "print(f'cuda_available={torch.cuda.is_available()}')\n"
            "torch.cuda.init()\n"
            "print(f'cuda_device_count={torch.cuda.device_count()}')\n"
            "print(f'cuda_device_name={torch.cuda.get_device_name(0)}')\n"
        )

        try:
            subprocess.run(
                [str(self.python_executable()), "-c", script],
                capture_output=True,
                check=True,
                env=self.server_environment(),
                text=True,
                timeout=60,
            )
        except subprocess.CalledProcessError as exception:
            raise RuntimeError(
                "CUDA runtime preflight failed before vLLM launch.\n"
                "stdout:\n"
                f"{exception.stdout.strip()}\n"
                "stderr:\n"
                f"{exception.stderr.strip()}"
            ) from exception
        except subprocess.TimeoutExpired as exception:
            raise RuntimeError(
                "CUDA runtime preflight timed out before vLLM launch.\n"
                f"stdout:\n{str(exception.stdout or '').strip()}\n"
                f"stderr:\n{str(exception.stderr or '').strip()}"
            ) from exception

    def build_command(self) -> list[str]:
        
        command = [
            str(self.python_executable()),
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--seed",
            str(self.cfg.seed),
            "--model",
            self.cfg.model_path,
            "--served-model-name",
            self.cfg.served_model_name,
            "--host",
            self.cfg.host,
            "--port",
            str(self.cfg.server_port),
            "--tensor-parallel-size",
            str(self.cfg.tensor_parallel_size),
            "--gpu-memory-utilization",
            str(self.cfg.gpu_memory_utilization),
            "--dtype",
            "auto",
            "--kv-cache-dtype",
            "auto",
            "--load-format",
            "auto",
            "--max-model-len",
            str(self.cfg.max_model_len),
            "--max-num-seqs",
            str(self.cfg.max_num_seqs),
            "--performance-mode",
            "throughput",
            "--compilation-config",
            json.dumps(self.cfg.compilation_config),
            "--trust-remote-code",
            "--download-dir",
            str(self.cfg.huggingface_hub_cache),
        ]


        if self.cfg.max_logprobs >= 0:
            command.extend([
                "--max-logprobs",
                str(self.cfg.max_logprobs),
            ])

        if self.cfg.stream_interval > 0:
            command.extend([
                "--stream-interval",
                str(self.cfg.stream_interval),
            ])

        if self.cfg.enable_prefix_caching:
            command.append("--enable-prefix-caching")

        if self.cfg.enable_chunked_prefill:
            command.append("--enable-chunked-prefill")

        if self.cfg.disable_log_stats:
            command.append("--disable-log-stats")

        if self.cfg.enable_async_scheduling:
            command.append("--async-scheduling")

        if self.cfg.enable_vllm_auto_tool_choice:
            command.append("--enable-auto-tool-choice")

            if self.cfg.vllm_tool_call_parser:
                command.extend([
                    "--tool-call-parser",
                    self.cfg.vllm_tool_call_parser,
                ])

        if self.cfg.vllm_chat_template_content_format:
            command.extend([
                "--chat-template-content-format",
                self.cfg.vllm_chat_template_content_format,
            ])

        if self.cfg.max_num_batched_tokens > 0:
            command.extend([
                "--max-num-batched-tokens",
                str(self.cfg.max_num_batched_tokens),
            ])

        return command

    def wait_until_ready(self) -> None:
        
        deadline = time.time() + self.cfg.server_timeout
        last_error = ""

        while time.time() < deadline:
            if self.process is not None and self.process.poll() is not None:
                raise RuntimeError(self.exit_message())

            try:
                with urllib.request.urlopen(self.health_url(), timeout=5) as response:
                    if response.status == 200:
                        return
            except (urllib.error.URLError, TimeoutError) as exception:
                last_error = str(exception)

            time.sleep(1)

        raise RuntimeError(f"vLLM server did not become ready: {last_error}")

    def health_url(self) -> str:
        
        host = "127.0.0.1" if self.cfg.host == "0.0.0.0" else self.cfg.host

        return f"http://{host}:{self.cfg.server_port}/health"

    def server_environment(self) -> dict[str, str]:
        
        startup_path = self.cfg.output_root / "python_startup"
        startup_path.mkdir(parents=True, exist_ok=True)
        sitecustomize_path = startup_path / "sitecustomize.py"
        sitecustomize_path.write_text(
            textwrap.dedent(
                """
                import os

                if os.environ.get("AIMO_DISABLE_TORCH_COMPILE", "1").lower() in {"1", "true", "yes", "on"}:
                    try:
                        import sys
                        import types

                        import torch

                        compile_fx_module = types.ModuleType("torch._inductor.compile_fx")

                        def _aimo_compile_fx(*args, **kwargs):

                            raise RuntimeError("torch._inductor.compile_fx is disabled for this notebook run")

                        def _aimo_compile_fx_inner(*args, **kwargs):

                            raise RuntimeError("torch._inductor.compile_fx_inner is disabled for this notebook run")

                        def _aimo_graph_returns_tuple(graph):

                            return True

                        compile_fx_module.compile_fx = _aimo_compile_fx
                        compile_fx_module.compile_fx_inner = _aimo_compile_fx_inner
                        compile_fx_module.graph_returns_tuple = _aimo_graph_returns_tuple
                        sys.modules["torch._inductor.compile_fx"] = compile_fx_module

                        if hasattr(torch, "_inductor"):
                            setattr(torch._inductor, "compile_fx", compile_fx_module)

                        def _aimo_no_compile(function=None, *args, **kwargs):

                            if function is None:
                                def _aimo_decorator(inner_function):

                                    return inner_function

                                return _aimo_decorator

                            return function

                        torch.compile = _aimo_no_compile
                    except Exception:
                        pass
                """
            ).strip() + "\n",
            encoding="utf-8",
        )
        environment = os.environ.copy()
        environment.update(self.cfg.vllm_environment)
        pythonpath = environment.get("PYTHONPATH", "")
        pythonpath_parts = [
            str(startup_path),
            *[
                part
                for part in pythonpath.split(os.pathsep)
                if part
            ],
        ]
        environment["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(pythonpath_parts))

        return environment

    def exit_message(self) -> str:
        
        return_code = self.process.returncode if self.process is not None else "unknown"

        return "\n".join([
            "vLLM server exited before readiness.",
            f"return_code={return_code}",
            "stdout_tail:",
            self.tail(self.stdout_path),
            "stderr_tail:",
            self.tail(self.stderr_path),
        ])

    def tail(self, path: Path, max_bytes: int = 20000) -> str:
        
        if not path.exists():
            return f"missing log file: {path}"

        with path.open("rb") as input_file:
            input_file.seek(0, os.SEEK_END)
            size = input_file.tell()
            input_file.seek(max(0, size - max_bytes), os.SEEK_SET)
            payload = input_file.read()

        return payload.decode("utf-8", errors="replace")

    def port_available(self, host: str, port: int) -> bool:
        
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as socket_object:
                socket_object.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                socket_object.bind((host, port))

            return True
        except OSError:
            return False

In [ ]:
class AIMOInferenceClient:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.client = OpenAI(
            base_url=self.cfg.base_url,
            api_key=self.cfg.api_key,
            timeout=self.cfg.request_timeout,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.cfg.model_path,
            trust_remote_code=self.cfg.trust_remote_code,
        )

    def chat_completion(
        self,
        messages: list[dict[str, Any]],
        tools: list[dict[str, Any]],
        max_tokens: int,
    ) -> AIMOChatResponse:
        
        input_tokens = self.count_chat_tokens(messages, tools)
        available_tokens = self.available_chat_output_tokens(input_tokens)
        resolved_max_tokens = self.resolve_output_token_count(available_tokens, max_tokens)
        payload = self.chat_sampling_payload(resolved_max_tokens)
        try:
            completion = self.client.chat.completions.create(
                messages=messages,
                **payload,
            )
        except Exception:
            return self.prompt_completion(
                messages=messages,
                tools=tools,
                max_tokens=max_tokens,
            )

        message = completion.choices[0].message
        response_tool_calls = list(getattr(message, "tool_calls", None) or [])
        raw_tool_calls = self.raw_response_tool_calls(response_tool_calls)
        tool_calls = self.parse_response_tool_calls(response_tool_calls, raw_tool_calls)

        return AIMOChatResponse(
            content=str(getattr(message, "content", "") or ""),
            tool_calls=tool_calls,
            raw_tool_calls=raw_tool_calls,
        )

    def prompt_completion(
        self,
        messages: list[dict[str, Any]],
        tools: list[dict[str, Any]],
        max_tokens: int,
    ) -> AIMOChatResponse:
        
        prompt = self.render_chat_prompt(messages, tools)
        stop_strings = [
            "</function_calls>",
            *self.cfg.stop_strings,
        ] if tools else self.cfg.stop_strings
        generated_text = self.stream_completion(
            prompt=prompt,
            max_tokens=max_tokens,
            stop_strings=stop_strings,
        )

        return AIMOChatResponse(
            content=generated_text,
            tool_calls=[],
            raw_tool_calls=[],
        )

    def render_chat_prompt(self, messages: list[dict[str, Any]], tools: list[dict[str, Any]]) -> str:
        
        try:
            if tools:
                return str(
                    self.tokenizer.apply_chat_template(
                        messages,
                        tools=tools,
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                )

            return str(
                self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )
        except Exception:
            return self.render_manual_chat_prompt(messages, tools)

    def render_manual_chat_prompt(self, messages: list[dict[str, Any]], tools: list[dict[str, Any]]) -> str:
        
        parts = []

        for message in messages:
            role = str(message.get("role", "user"))
            content = self.escape_chat_content(str(message.get("content", "")))

            if role == "system":
                parts.append("<|im_start|>system\n")
                parts.append(content)

                if tools:
                    parts.append("\n<functions>\n")
                    parts.append(self.manual_function_schema(tools))
                    parts.append("\n</functions>")

                parts.append("<|im_end|>\n")
                continue

            if role == "tool":
                role = "environment"

            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")

        parts.append("<|im_start|>assistant\n<think>")

        return "".join(parts)

    def manual_function_schema(self, tools: list[dict[str, Any]]) -> str:
        
        functions = []

        for tool in tools:
            function = tool.get("function", tool)

            if isinstance(function, dict):
                functions.append(function)

        return json.dumps(functions, ensure_ascii=False, separators=(",", ":"))

    def escape_chat_content(self, content: str) -> str:
        
        return (
            content.replace("<|im_start|>", "<|escaped_im_start|>")
            .replace("<|im_end|>", "<|escaped_im_end|>")
        )

    def chat_sampling_payload(self, max_tokens: int) -> dict[str, Any]:
        
        payload = self.sampling_payload(
            max_tokens=max_tokens,
            stop_strings=self.cfg.stop_strings,
        )
        payload.pop("logprobs", None)

        return payload

    def raw_response_tool_calls(self, response_tool_calls: Sequence[Any]) -> list[dict[str, Any]]:
        
        raw_tool_calls = []

        for index, response_tool_call in enumerate(response_tool_calls, start=1):
            function = getattr(response_tool_call, "function", None)
            name = str(getattr(function, "name", ""))
            arguments = getattr(function, "arguments", "")
            parsed_arguments = self.tool_arguments_payload(arguments)
            tool_call_id = str(getattr(response_tool_call, "id", "") or f"python_call_{index}")

            raw_tool_calls.append({
                "id": tool_call_id,
                "type": str(getattr(response_tool_call, "type", "function") or "function"),
                "function": {
                    "name": name,
                    "arguments": parsed_arguments,
                },
            })

        return raw_tool_calls

    def tool_arguments_payload(self, arguments: object) -> dict[str, str]:
        
        code = self.code_from_tool_arguments(arguments)

        return {"code": code} if code else {}

    def parse_response_tool_calls(
        self,
        response_tool_calls: Sequence[Any],
        raw_tool_calls: list[dict[str, Any]],
    ) -> list[AIMOToolCall]:
        
        tool_calls = []

        for response_tool_call, raw_tool_call in zip(response_tool_calls, raw_tool_calls):
            function = getattr(response_tool_call, "function", None)
            name = str(getattr(function, "name", ""))
            arguments = getattr(function, "arguments", "")
            code = self.code_from_tool_arguments(arguments)

            if name == "python" and code:
                tool_calls.append(AIMOToolCall(
                    name="python",
                    code=code,
                    tool_call_id=str(raw_tool_call["id"]),
                ))

        return tool_calls

    def code_from_tool_arguments(self, arguments: object) -> str:
        
        if isinstance(arguments, dict):
            return str(arguments.get("code", "")).strip()

        arguments_text = str(arguments or "").strip()

        if not arguments_text:
            return ""

        try:
            payload = json.loads(arguments_text)
        except json.JSONDecodeError:
            return arguments_text

        if isinstance(payload, dict):
            return str(payload.get("code", "")).strip()

        return ""

    def stream_completion(
        self,
        prompt: str,
        max_tokens: int,
        stop_strings: Sequence[str] | None = None,
        watch_final: bool = True,
    ) -> str:
        
        resolved_max_tokens = self.resolve_output_tokens(prompt, max_tokens)
        payload = self.sampling_payload(
            max_tokens=resolved_max_tokens,
            stop_strings=stop_strings,
        )
        stream = self.client.completions.create(
            prompt=prompt,
            stream=True,
            **payload,
        )
        chunks = []
        generated_text = ""

        try:
            for chunk in stream:
                text = str(chunk.choices[0].text or "")

                if text:
                    chunks.append(text)
                    generated_text += text
        finally:
            stream.close()

        return "".join(chunks)

    def sampling_payload(
        self,
        max_tokens: int,
        stop_strings: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        
        payload = {
            "model": self.cfg.served_model_name,
            "max_tokens": max_tokens,
            "temperature": self.cfg.temperature,
            "stop": list(stop_strings) if stop_strings is not None else self.cfg.stop_strings,
        }
        extra_body = {}

        if self.cfg.request_logprobs >= 0:
            payload["logprobs"] = self.cfg.request_logprobs

        if self.cfg.top_p > 0:
            payload["top_p"] = self.cfg.top_p

        if self.cfg.top_k > 0:
            extra_body["top_k"] = self.cfg.top_k

        if self.cfg.min_p > 0:
            extra_body["min_p"] = self.cfg.min_p

        if self.cfg.presence_penalty != 0.0:
            payload["presence_penalty"] = self.cfg.presence_penalty

        if self.cfg.repetition_penalty != 1.0:
            extra_body["repetition_penalty"] = self.cfg.repetition_penalty

        if extra_body:
            payload["extra_body"] = extra_body

        return payload

    def resolve_output_tokens(self, prompt: str, requested_max_tokens: int) -> int:
        
        available_tokens = self.available_output_tokens(prompt)

        return self.resolve_output_token_count(available_tokens, requested_max_tokens)

    def resolve_output_token_count(self, available_tokens: int, requested_max_tokens: int) -> int:
        
        resolved_max_tokens = min(
            available_tokens,
            self.cfg.max_generation_tokens,
            max(0, requested_max_tokens),
        )

        if resolved_max_tokens <= 0:
            raise ValueError("No output tokens are available for generation.")

        return resolved_max_tokens

    def available_output_tokens(self, prompt: str) -> int:
        
        input_tokens = self.count_tokens(prompt)
        available_tokens = self.cfg.max_model_len - input_tokens

        if available_tokens <= 0:
            raise ValueError(f"Prompt uses {input_tokens} tokens and exceeds max_model_len={self.cfg.max_model_len}.")

        return available_tokens

    def count_chat_tokens(self, messages: list[dict[str, Any]], tools: list[dict[str, Any]]) -> int:
        
        try:
            if tools:
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tools=tools,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            else:
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        except Exception:
            prompt = self.render_manual_chat_prompt(messages, tools)

        return self.count_tokens(str(prompt))

    def available_chat_output_tokens(self, input_tokens: int) -> int:
        
        available_tokens = self.cfg.max_model_len - input_tokens

        if available_tokens <= 0:
            raise ValueError(f"Prompt uses {input_tokens} tokens and exceeds max_model_len={self.cfg.max_model_len}.")

        return available_tokens

    def count_tokens(self, text: str) -> int:
        
        return len(self.tokenizer.encode(text, add_special_tokens=False))

In [ ]:
class AIMOPromptBuilder:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg

    def first_pass_messages(self, problem: str) -> list[AIMOChatMessage]:
        
        return self.messages(
            system_prompt=self.cfg.first_pass_system_prompt,
            user_prompt=self.cfg.first_pass_prompt.format(problem=problem.strip()),
        )

    def second_pass_messages(
        self,
        problem: str,
        first_solution: str,
    ) -> list[AIMOChatMessage]:
        
        return self.messages(
            system_prompt=self.cfg.second_pass_system_prompt,
            user_prompt=self.cfg.second_pass_prompt.format(
                problem=problem.strip(),
                first_solution=first_solution.strip() or "No final answer was produced.",
            ),
        )

    def messages(self, system_prompt: str, user_prompt: str) -> list[AIMOChatMessage]:
        
        return [
            AIMOChatMessage(role="system", content=system_prompt),
            AIMOChatMessage(role="user", content=user_prompt.strip()),
        ]

In [ ]:
class AIMOSolver:
    
    def __init__(self, cfg: CFG, client: AIMOInferenceClient, sandbox: AIMOSandbox) -> None:
        
        self.cfg = cfg
        self.client = client
        self.sandbox = sandbox
        self.template = AIMOChatMLTemplate(client.tokenizer)
        self.prompt_builder = AIMOPromptBuilder(cfg)

    def solve_problem(self, record: AIMOProblemRecord) -> AIMOProblemResult:
        
        first_pass = self.run_pass(
            name="solve",
            messages=self.prompt_builder.first_pass_messages(record.problem),
        )
        second_pass = self.run_pass(
            name="audit",
            messages=self.prompt_builder.second_pass_messages(
                problem=record.problem,
                first_solution=first_pass.final_text,
            ),
        )
        return AIMOProblemResult(
            record=record,
            passes=[first_pass, second_pass],
        )

    def run_pass(
        self,
        name: str,
        messages: list[AIMOChatMessage],
    ) -> AIMOPassResult:
        
        started_at = time.time()
        deadline = started_at + self.cfg.pass_timeout
        transcript_messages = [
            self.template.message_payload(message)
            for message in messages
        ]
        tools = self.template.tool_schema(self.cfg.tool_prompt)
        implicit_think_opened = "think" in str(self.cfg.served_model_name).lower() or "think" in str(self.cfg.model_path).lower()
        raw_chunks = ["<think>"] if implicit_think_opened else []
        tool_outputs = []
        python_calls = 0
        python_errors = 0
        input_tokens = 0
        output_tokens = 0

        for _ in range(self.cfg.max_tool_rounds):
            if self.remaining_seconds(deadline) <= 0:
                break

            if self.remaining_generation_tokens(output_tokens) <= self.cfg.minimum_available_tokens:
                break

            input_tokens += self.client.count_chat_tokens(transcript_messages, tools)
            chat_response = self.generate_chat(
                messages=transcript_messages,
                tools=tools,
                output_tokens=output_tokens,
            )
            generated_text = chat_response.content
            output_tokens += self.client.count_tokens(generated_text)
            tool_calls = chat_response.tool_calls
            raw_tool_calls = chat_response.raw_tool_calls

            if tools and not tool_calls:
                tool_calls = self.template.parse_tool_calls(generated_text)
                tool_calls, raw_tool_calls = self.normalize_tool_call_ids(tool_calls, python_calls)

            if tool_calls and generated_text:
                generated_text = self.template.normalize_tool_text(generated_text)

            raw_chunks.append(generated_text)
            assistant_message = {
                "role": "assistant",
                "content": generated_text,
            }

            if raw_tool_calls:
                assistant_message["tool_calls"] = raw_tool_calls

            transcript_messages.append(assistant_message)

            if not tool_calls:
                break

            environment_outputs, tool_messages, python_calls, python_errors = self.execute_tool_calls(
                tool_calls=tool_calls,
                tool_outputs=tool_outputs,
                python_calls=python_calls,
                python_errors=python_errors,
            )

            if not environment_outputs:
                break

            transcript_messages.extend(tool_messages)

        raw_text = "\n".join(raw_chunks)
        final_text = self.template.final_channel(raw_text)

        self.print_final_solution(name, final_text)

        return AIMOPassResult(
            name=name,
            final_text=final_text,
            tool_output="\n\n".join(tool_outputs),
            python_calls=python_calls,
            python_errors=python_errors,
            elapsed_seconds=time.time() - started_at,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
        )

    def generate_chat(
        self,
        messages: list[dict[str, Any]],
        tools: list[dict[str, Any]],
        output_tokens: int,
    ) -> AIMOChatResponse:
        
        return self.client.prompt_completion(
            messages=messages,
            tools=tools,
            max_tokens=self.remaining_generation_tokens(output_tokens),
        )

    def normalize_tool_call_ids(
        self,
        tool_calls: list[AIMOToolCall],
        python_calls: int,
    ) -> tuple[list[AIMOToolCall], list[dict[str, Any]]]:
        
        normalized_tool_calls = []
        raw_tool_calls = []

        for index, tool_call in enumerate(tool_calls, start=1):
            tool_call_id = tool_call.tool_call_id or f"python_call_{python_calls + index}"
            normalized_tool_call = AIMOToolCall(
                name=tool_call.name,
                code=tool_call.code,
                tool_call_id=tool_call_id,
            )
            normalized_tool_calls.append(normalized_tool_call)
            raw_tool_calls.append({
                "id": tool_call_id,
                "type": "function",
                "function": {
                    "name": normalized_tool_call.name,
                    "arguments": {"code": normalized_tool_call.code},
                },
            })

        return normalized_tool_calls, raw_tool_calls

    def execute_tool_calls(
        self,
        tool_calls: list[AIMOToolCall],
        tool_outputs: list[str],
        python_calls: int,
        python_errors: int,
    ) -> tuple[list[str], list[dict[str, Any]], int, int]:
        
        environment_outputs = []
        tool_messages = []

        for tool_call in tool_calls:
            if python_calls >= self.cfg.max_python_calls:
                break

            result = self.sandbox.execute(tool_call.code)
            python_calls += 1
            python_errors += 0 if result.success else 1
            tool_payload = self.format_tool_output(python_calls, tool_call, result)
            tool_outputs.append(tool_payload)
            environment_outputs.append(tool_payload)
            tool_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.tool_call_id,
                "content": tool_payload,
            })

        return environment_outputs, tool_messages, python_calls, python_errors

    def print_final_solution(self, name: str, final_text: str) -> None:
        
        print("", flush=True)
        print(f"{name.title()} Final Channel:")
        print(final_text.strip() or "No final channel was produced.")
        print("", flush=True)

    def remaining_generation_tokens(self, output_tokens: int) -> int:
        
        return max(0, self.cfg.max_generation_tokens - output_tokens)

    def remaining_seconds(self, deadline: float) -> float:
        
        return deadline - time.time()

    def format_tool_output(self, index: int, tool_call: AIMOToolCall, result: AIMOToolResult) -> str:
        
        return (
            f"Python call {index}\n"
            f"Code:\n{tool_call.code}\n"
            f"Output:\n{result.to_payload()}"
        )

In [ ]:
class AIMORunner:
    
    def __init__(self, cfg: CFG) -> None:
        
        self.cfg = cfg
        self.environment = AIMOEnvironment(cfg)
        self.reader = AIMOInputReader(cfg)
        self.writer = AIMOOutputWriter(cfg)

    def run(self) -> None:
        
        with AIMOOutputPulse(self.cfg.output_ping_interval, self.cfg.output_pulse_enabled):
            self.run_with_pulse()

    def run_with_pulse(self) -> None:
        
        self.environment.configure()
        records = self.reader.read_records()
        results = []

        with AIMOServer(self.cfg):
            client = AIMOInferenceClient(self.cfg)
            sandbox = AIMOSandbox(self.cfg)
            solver = AIMOSolver(self.cfg, client, sandbox)

            try:
                for record in records:
                    result = solver.solve_problem(record)
                    results.append(result)
                    self.writer.write_results(results)
                    sandbox.reset()
            finally:
                sandbox.close()

        self.writer.write_results(results)

In [ ]:
runner = AIMORunner(CFG)
runner.run()